In [6]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import os
import re
from datetime import datetime

In [15]:
def clean_log_file(file_path):
    # Initialize lists to store data
    timestamps = []
    data_arrays = []
    is_release = []
    is_landing = []
    
    # Get the file name without extension
    file_name = os.path.basename(file_path).split('.')[0]
    
    # Read the file line by line
    with open(file_path, 'r') as file:
        for line in file:
            # Skip empty lines
            if not line.strip():
                continue
            
            # Check for release event
            if "Release at" in line:
                release_match = re.search(r'Release at (\d+)', line)
                if release_match:
                    release_time = int(release_match.group(1))
                    # Add a row with release event
                    timestamps.append(datetime.fromtimestamp(release_time/1000).isoformat())
                    data_arrays.append([np.nan] * 100)  # Placeholder array with NaN values
                    is_release.append(1)
                    is_landing.append(0)
                continue
            
            # Check for landing event
            if "Landing at" in line:
                landing_match = re.search(r'Landing at (\d+)', line)
                if landing_match:
                    landing_time = int(landing_match.group(1))
                    # Add a row with landing event
                    timestamps.append(datetime.fromtimestamp(landing_time/1000).isoformat())
                    data_arrays.append([np.nan] * 100)  # Placeholder array with NaN values
                    is_release.append(0)
                    is_landing.append(1)
                continue
            
            # Look for data lines with timestamps and arrays
            if "LOG" in line and "[" in line and "]" in line:
                # Extract timestamp
                timestamp_match = re.search(r'\[(.*?)\]', line)
                if timestamp_match:
                    timestamp = timestamp_match.group(1)
                    
                    # Extract the array of numbers
                    array_match = re.search(r'\[(.*?)\]$', line)
                    if array_match:
                        # Convert string of numbers to list of floats
                        try:
                            numbers = [float(x.strip()) for x in array_match.group(1).split(',')]
                            timestamps.append(timestamp)
                            data_arrays.append(numbers)
                            is_release.append(0)
                            is_landing.append(0)
                        except ValueError:
                            continue

    # Convert to DataFrame
    df = pd.DataFrame(data_arrays, columns=[f'value_{i}' for i in range(len(data_arrays[0]))])
    
    # Add timestamp column using mixed format to handle different timestamp formats
    df['timestamp'] = pd.to_datetime(timestamps, format='mixed')
    
    # Extract only the time component
    df['time'] = df['timestamp'].dt.time
    
    # Add file name column
    df['file_name'] = file_name
    
    # Add release and landing columns
    df['is_release'] = is_release
    df['is_landing'] = is_landing
    
    # Reorder columns to have time, file_name, and event columns first
    cols = ['time', 'file_name', 'is_release', 'is_landing'] + [col for col in df.columns if col not in ['timestamp', 'time', 'file_name', 'is_release', 'is_landing']]
    df = df[cols]
    
    return df

In [16]:
file_path = "txt_files/cleaned/OwenDummy.txt"

owen_dummy = clean_log_file(file_path)

In [17]:
owen_dummy

,time,file_name,is_release,is_landing,value_0,value_1,value_2,value_3,value_4,value_5,...,value_90,value_91,value_92,value_93,value_94,value_95,value_96,value_97,value_98,value_99
0,19:44:50.216000,OwenDummy,0,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,19:44:50.688000,OwenDummy,1,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,19:44:51.172000,OwenDummy,0,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,19:44:52.217000,OwenDummy,1,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,19:44:52.314000,OwenDummy,0,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,19:45:43.804000,OwenDummy,1,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
72,19:45:43.953000,OwenDummy,0,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
73,19:45:44.072000,OwenDummy,1,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74,19:45:44.664000,OwenDummy,0,1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
